# PortoFlow — kiểm toán khoảng trống độ phủ

Notebook này **chỉ chẩn đoán metadata/QC**. Không matching, không đọc ledger13GB,
không nhập holdout vào train, không đổi profile hoặc màu bản đồ. Báo cáo xong
chưa có nghĩa độ phủ tăng hoặc tìm được parent có thể khôi phục.

**Input duy nhất:** Add Input → Notebook Outputs → `anhthu128/porto-flow`,
đúng Saved Version matching ban đầu có `_output_.zip` khoảng5,48GB.
Không chọn `porto-flow-v2` hoặc Output recovery/report. Không cần raw Porto,
code Dataset hay pip install. CPU/None, Internet **Off**, giữ notebook **Private**.

Ngân sách:45phút gồm kiểm SHA toàn ZIP, RAM mềm1,5GiB, Output audit mềm768MiB,
reserve256MiB, tối đa100.000batch mới, không giải nén hoặc tạo scratch ledger.
Giới hạn được kiểm trước mỗi batch; tối đa một batch metadata có thể vượt soft cap.
Một job mỗi lần. Đây là trần, không phải lời hứa thời gian hoàn tất.

Lượt này kiểm parent IDs/source rows trùng, checksum `parents.json`, native status,
block-QC và đối chiếu tổng parent với summary. Không kiểm lại Parquet/GPS độc lập.
Chỉ lấy tối đa100ví dụ train chưa có quan sát hợp lệ; holdout chỉ đếm đối chiếu.

**Chạy:** Save Version → Save & Run All. Gửi `portoflow_coverage_audit_review.zip`;
giữ cả thư mục Output `portoflow_coverage_audit` để có state tiếp tục khi hết giờ.
Nếu PARTIAL: attach Output nhỏ lần này và cùng Input matching gốc, đặt RESUME tới
thư mục audit ở Input trước khi chạy tiếp. Không chạy lại pipeline6giờ.


In [ ]:
from pathlib import Path
import hashlib, importlib.util, json, shutil, zipfile

SOURCE = None  # Optional explicit original _output_.zip path when auto-detection is ambiguous.
RESUME = None  # Optional attached prior SMALL audit directory, not a recovery ledger.
WORK = Path('/kaggle/working/portoflow_coverage_audit')
SCRIPT_PATH = Path('/kaggle/working/portoflow_parent_qc_audit.py')
SCRIPT = '"""Bounded, resumable metadata-only QC audit of the ORIGINAL saved PortoFlow ZIP.\n\nStandard library only. Never extracts the ZIP, reads observation Parquet, uses\nthe recovery ledger, rematches GPS, changes profiles or activates an export.\n"""\nimport argparse\nfrom collections import Counter\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path, PurePosixPath\nimport shutil\nimport sqlite3\nimport time\nimport zipfile\n\nVERSION = \'PORTOFLOW_PARENT_QC_AUDIT_V1\'\nSOURCE_SHA = \'71e0a5ee45f1a241835b32254068e9009983d4c5067ef8f4edc821840803a93b\'\nCORE_GRAPH = \'401fea1a982be227710b427ea651784d87d78b0ded45433c5c8f72866b385031\'\nMiB = 1024**2\nDEFAULT_LIMITS = dict(seconds=2700, output_bytes=768*MiB, reserve_bytes=256*MiB,\n                      rss_mb=1536, max_new_batches=100000)\nALLOWED_OUTPUTS = {\'audit_state.sqlite\', \'audit_state.sqlite-journal\', \'summary.json\',\n                   \'examples.json\', \'budget.json\', \'last_attempt.json\'}\n\n\ndef encode(value):\n    return (json.dumps(value, ensure_ascii=False, sort_keys=True, indent=2, allow_nan=False)+\'\\n\').encode()\n\n\ndef digest(data):\n    return hashlib.sha256(data).hexdigest()\n\n\ndef atomic_json(path, value):\n    path = Path(path)\n    temporary = path.with_name(\'.\'+path.name+\'.tmp\')\n    temporary.write_bytes(encode(value))\n    os.replace(temporary, path)\n\n\nclass Budget:\n    def __init__(self, work, limits):\n        self.work, self.limits = work, limits\n        self.deadline = time.monotonic()+limits[\'seconds\']-min(30, limits[\'seconds\']/10)\n\n    def check(self):\n        if time.monotonic() >= self.deadline:\n            raise TimeoutError(\'Finite audit deadline reached; committed batches retained\')\n        size = sum(p.stat().st_size for p in self.work.iterdir() if p.is_file())\n        if size > self.limits[\'output_bytes\']:\n            raise OSError(\'Audit Output cap reached; committed batches retained\')\n        if shutil.disk_usage(self.work).free < self.limits[\'reserve_bytes\']:\n            raise OSError(\'Audit disk reserve reached; committed batches retained\')\n        status = Path(\'/proc/self/status\')\n        if status.is_file():\n            for line in status.read_text().splitlines():\n                if line.startswith(\'VmRSS:\') and int(line.split()[1]) > self.limits[\'rss_mb\']*1024:\n                    raise MemoryError(\'Audit soft RAM ceiling reached; committed batches retained\')\n\n\ndef sha_file(path, check=lambda: None):\n    h = hashlib.sha256(); size = 0; announced = 0\n    with Path(path).open(\'rb\') as stream:\n        while True:\n            check(); data = stream.read(8*MiB)\n            if not data: break\n            h.update(data); size += len(data)\n            if size-announced >= 512*MiB:\n                print(\'VERIFY original ZIP:\', round(size/MiB), \'MiB read\', flush=True); announced = size\n    return h.hexdigest()\n\n\nclass SavedZip:\n    def __init__(self, path):\n        self.archive = zipfile.ZipFile(path)\n        try:\n            names = {}\n            for info in self.archive.infolist():\n                parts = PurePosixPath(info.filename)\n                if parts.is_absolute() or \'..\' in parts.parts or \'\\\\\' in info.filename:\n                    raise ValueError(\'Unsafe archive member\')\n                name = parts.as_posix()\n                if name in names: raise ValueError(\'Duplicate canonical ZIP member\')\n                names[name] = info\n            roots = [n[:-len(\'matching_summary.json\')] for n in names if n.endswith(\'/matching_summary.json\') or n == \'matching_summary.json\']\n            roots = [r for r in roots if r+\'graph/graph_metadata.json\' in names]\n            preferred = [r for r in roots if r.rstrip(\'/\').endswith(\'portoflow_run\')]\n            roots = preferred or roots\n            if len(roots) != 1: raise ValueError(\'Select original matching Output with exactly one PortoFlow run\')\n            prefix = roots[0]\n            self.members = {n[len(prefix):]: info for n,info in names.items() if n.startswith(prefix)}\n            self.ids = sorted({n.split(\'/\')[1] for n in self.members\n                if n.startswith(\'region_batches/\') and n.endswith(\'/run_metadata.json\') and not n.split(\'/\')[1].startswith(\'.\')},\n                key=lambda bid:self.members[\'region_batches/\'+bid+\'/run_metadata.json\'].header_offset)\n            if not self.ids: raise ValueError(\'No committed parent batches\')\n        except Exception:\n            self.archive.close(); raise\n\n    def read(self, name):\n        info = self.members[name]\n        if info.file_size > 16*MiB: raise ValueError(\'Audit metadata member exceeds16MiB cap\')\n        return self.archive.read(info)  # CRC checked by zipfile.\n\n    def json(self, name):\n        return json.loads(self.read(name))\n\n    def close(self):\n        self.archive.close()\n\n\ndef parent_reason(row):\n    if row[\'accepted_observations\'] > 0: return \'has_accepted_core_observations\'\n    blocks = row[\'blocks\']\n    if not blocks: return \'no_saved_candidate_blocks\'\n    if all(b[\'native_status\'] not in {\'success\',\'partial\'} for b in blocks):\n        return \'all_native_blocks_unsuccessful\'\n    if all(b.get(\'qc\',{}).get(\'accepted_gaps\',0) == 0 for b in blocks):\n        return \'no_accepted_gaps_in_saved_block_qc\'\n    return \'block_qc_accepted_gaps_but_no_accepted_core_observations\'\n\n\ndef batch_statistics(parents):\n    metrics = Counter(); examples = []\n    for row in parents:\n        if (row[\'split\'] not in {\'train\',\'holdout\'} or not isinstance(row[\'trip_id\'],str)\n                or not 0 < len(row[\'trip_id\']) <= 128 or type(row[\'source_row\']) is not int\n                or row[\'source_row\'] < 0 or type(row[\'accepted_observations\']) is not int\n                or row[\'accepted_observations\'] < 0 or type(row[\'whole_parent_complete\']) is not bool):\n            raise ValueError(\'Invalid saved parent record\')\n        split = row[\'split\']\n        metrics[(split,\'parents\')] += 1\n        metrics[(split,\'parents_with_accepted_core_observations\')] += int(row[\'accepted_observations\'] > 0)\n        metrics[(split,\'whole_parent_complete\')] += int(row[\'whole_parent_complete\'])\n        metrics[(split,\'accepted_observation_rows_reported\')] += row[\'accepted_observations\']\n        if split != \'train\': continue  # Holdout is bookkeeping, never a recovery target.\n        reason = parent_reason(row)\n        metrics[(split,\'parent_reason/\'+reason)] += 1\n        for block in row[\'blocks\']:\n            metrics[(split,\'native_status/\'+str(block[\'native_status\']))] += 1\n            qc = block.get(\'qc\',{})\n            if qc.get(\'rejection\'): metrics[(split,\'qc_rejection/\'+str(qc[\'rejection\']))] += 1\n            for label,count in qc.get(\'rejected_gaps\',{}).items():\n                if type(count) is not int or count < 0: raise ValueError(\'Invalid rejected-gap count\')\n                metrics[(split,\'rejected_gap_reason/\'+str(label))] += count\n        if row[\'accepted_observations\'] == 0:\n            examples.append({\'reason\':reason, **row})\n    return metrics, examples\n\n\ndef open_state(work, identity):\n    db = sqlite3.connect(work/\'audit_state.sqlite\')\n    db.execute(\'PRAGMA journal_mode=DELETE\'); db.execute(\'PRAGMA cache_size=-16384\')\n    db.executescript(\'\'\'\n        CREATE TABLE IF NOT EXISTS meta(key TEXT PRIMARY KEY,value TEXT);\n        CREATE TABLE IF NOT EXISTS batches(batch TEXT PRIMARY KEY,receipt TEXT);\n        CREATE TABLE IF NOT EXISTS parents(trip_id TEXT PRIMARY KEY,source_row INTEGER UNIQUE,split TEXT);\n        CREATE TABLE IF NOT EXISTS metrics(split TEXT,label TEXT,n INTEGER,PRIMARY KEY(split,label));\n        CREATE TABLE IF NOT EXISTS examples(trip_id TEXT PRIMARY KEY,reason TEXT,record TEXT);\n    \'\'\')\n    stored = db.execute("SELECT value FROM meta WHERE key=\'identity\'").fetchone()\n    fingerprint = digest(encode(identity))\n    if stored and stored[0] != fingerprint:\n        db.close(); raise ValueError(\'Resume audit state belongs to another source/tool identity\')\n    with db:\n        db.execute("INSERT OR IGNORE INTO meta VALUES (\'identity\',?)",(fingerprint,))\n    return db\n\n\ndef snapshot(db, matching, status, phase, error, source_sha, limits, batches_total):\n    values = {split:dict(db.execute(\'SELECT label,n FROM metrics WHERE split=?\',(split,)))\n              for split in (\'train\',\'holdout\')} if db else {}\n    return {\'status\':status,\'phase\':phase,\'error\':error,\'source_zip_sha256\':source_sha,\n        \'tool_version\':VERSION,\'budget\':limits,\'committed_batches\':db.execute(\'SELECT COUNT(*) FROM batches\').fetchone()[0] if db else 0,\n        \'available_batches\':batches_total,\'metrics\':values,\n        \'matching_completed_parents_reported\':matching.get(\'completed_parents\') if matching else None,\n        \'matching_expected_parents_reported\':matching.get(\'expected_parents\') if matching else None,\n        \'matching_complete_reported\':matching.get(\'all_eligible_complete\') if matching else None,\n        \'coverage_changed\':False,\'can_estimate_additional_coverage\':False,\n        \'interpretation\':\'Parent/block QC metadata audit only; zero accepted core observations is not proof of a matcher bug.\',\n        \'limits\':[\'No observation Parquet, raw GPS, sealed ledger or matching is read or executed.\',\n            \'Block accepted_gaps is before later trimming/clipping/speed filtering; not a causal diagnosis.\',\n            \'No rejected parent is accepted or recolored by this tool.\',\n            \'Holdout counts are bookkeeping only; examples/recovery targets are train only.\']}\n\n\ndef audit(source, work, *, limits=None, expected_sha=SOURCE_SHA, expected_graph=CORE_GRAPH):\n    limits = {**DEFAULT_LIMITS, **(limits or {})}\n    if any(type(value) is not int or value <= 0 for value in limits.values()):\n        raise ValueError(\'Finite positive integer audit limits required\')\n    source,work = Path(source).resolve(),Path(work).resolve()\n    if not source.is_file() or source.stat().st_size > 8*1024**3:\n        raise ValueError(\'Attach original matching Output ZIP, maximum8GiB; do not select recovery ledger\')\n    if source.is_relative_to(work): raise ValueError(\'Audit Output cannot contain the original source\')\n    work.mkdir(parents=True,exist_ok=True)\n    if any(p.name not in ALLOWED_OUTPUTS for p in work.iterdir()):\n        raise ValueError(\'Use a dedicated audit Output directory, never an app export or matching run\')\n    atomic_json(work/\'budget.json\',limits)\n    budget=Budget(work,limits);db=saved=None;matching=None;batch_total=0;phase=\'source_sha\';actual=None\n    status=\'PARENT_QC_AUDIT_FAILED\';error=None\n    try:\n        budget.check();actual=sha_file(source,budget.check)\n        if actual != expected_sha: raise ValueError(\'Original Output ZIP SHA differs; select the pinned initial matching version\')\n        phase=\'source_metadata\';saved=SavedZip(source);budget.check()\n        matching=saved.json(\'matching_summary.json\');identity=matching[\'identity\']\n        source_summary=saved.json(\'source_summary.json\')\n        if source_summary[\'status\'] != \'SOURCE_EOF_COMPLETE\' or source_summary[\'identity\'] != identity[\'source\']:\n            raise ValueError(\'Source EOF/identity mismatch\')\n        if identity[\'core_graph\'] != expected_graph or saved.json(\'graph/graph_metadata.json\')[\'graph_version\'] != expected_graph:\n            raise ValueError(\'Core graph mismatch\')\n        batch_total=len(saved.ids)\n        state_file=work/\'audit_state.sqlite\'\n        if state_file.exists():\n            previous_summary=json.loads((work/\'summary.json\').read_bytes())\n            if sha_file(state_file,budget.check) != previous_summary.get(\'audit_state_sha256\'):\n                raise ValueError(\'Resume audit-state seal differs; keep original Output and use a fresh audit directory\')\n        db=open_state(work,{\'source_sha\':actual,\'matching\':identity,\'version\':VERSION,\n                           \'auditor_sha\':sha_file(Path(__file__))})\n        phase=\'parent_batches\';processed=0\n        for bid in saved.ids:\n            budget.check()\n            previous=db.execute(\'SELECT receipt FROM batches WHERE batch=?\',(bid,)).fetchone()\n            # Every resume re-hashes the entire pinned original ZIP and tool identity.\n            if previous: continue\n            if processed >= limits[\'max_new_batches\']: raise TimeoutError(\'Finite per-run batch cap reached\')\n            base=\'region_batches/\'+bid+\'/\'\n            manifest_data=saved.read(base+\'run_metadata.json\');manifest=json.loads(manifest_data)\n            if manifest[\'identity\'] != identity: raise ValueError(\'Batch matching identity differs: \'+bid)\n            data=saved.read(base+\'parents.json\')\n            if digest(data) != manifest[\'outputs_sha256\'][\'parents.json\']:\n                raise ValueError(\'Saved parent checksum differs: \'+bid)\n            parents=json.loads(data)\n            if not isinstance(parents,list) or not parents: raise ValueError(\'Empty/invalid parent batch\')\n            metrics,examples=batch_statistics(parents)\n            with db:\n                try:\n                    db.executemany(\'INSERT INTO parents VALUES (?,?,?)\',[(r[\'trip_id\'],r[\'source_row\'],r[\'split\']) for r in parents])\n                except sqlite3.IntegrityError as exc:\n                    raise ValueError(\'Duplicate original parent ID/source row across batches\') from exc\n                for (split,label),n in metrics.items():\n                    db.execute(\'INSERT INTO metrics VALUES (?,?,?) ON CONFLICT(split,label) DO UPDATE SET n=n+excluded.n\',(split,label,n))\n                count=db.execute(\'SELECT COUNT(*) FROM examples\').fetchone()[0]\n                for example in examples:\n                    if count >= 100: break\n                    per_reason=db.execute(\'SELECT COUNT(*) FROM examples WHERE reason=?\',(example[\'reason\'],)).fetchone()[0]\n                    if per_reason >= 25: continue\n                    encoded=encode(example)\n                    if len(encoded) > 64*1024: continue\n                    db.execute(\'INSERT INTO examples VALUES (?,?,?)\',(example[\'trip_id\'],example[\'reason\'],encoded.decode()))\n                    count += 1\n                db.execute(\'INSERT INTO batches VALUES (?,?)\',(bid,digest(manifest_data+data)))\n            processed += 1\n            if processed%1000 == 0:\n                print(\'AUDIT committed\',db.execute(\'SELECT COUNT(*) FROM batches\').fetchone()[0],\'/\',batch_total,\'batches\',flush=True)\n        phase=\'reconciliation\'\n        for split in (\'train\',\'holdout\'):\n            values=dict(db.execute(\'SELECT label,n FROM metrics WHERE split=?\',(split,)))\n            if values.get(\'parents\',0) != matching[\'completed_parents\'].get(split,0):\n                raise ValueError(\'Parent total differs from saved matching summary: \'+split)\n            if values.get(\'parents_with_accepted_core_observations\',0) != matching[\'parents_with_accepted_core_observations\'].get(split,0):\n                raise ValueError(\'Accepted-parent total differs from saved matching summary: \'+split)\n        status=\'PARENT_QC_AUDIT_COMPLETE\';phase=\'complete\'\n    except (TimeoutError,MemoryError,OSError) as exc:\n        status=\'PARENT_QC_AUDIT_PARTIAL\';error=str(exc)\n    except Exception as exc:\n        error=type(exc).__name__+\': \'+str(exc)\n    finally:\n        result=snapshot(db,matching,status,phase,error,actual,limits,batch_total)\n        if matching:\n            result[\'data_kind\']=\'SYNTHETIC\' if matching[\'identity\'][\'source\'].get(\'synthetic\') else \'REAL_SAVED_METADATA_PENDING_RAW_QC\'\n            result[\'source_scan_inventory_reported\']=source_summary.get(\'inventory\') if \'source_summary\' in locals() else None\n        if db:\n            examples=[json.loads(row[0]) for row in db.execute(\'SELECT record FROM examples ORDER BY reason,trip_id\')]\n            atomic_json(work/\'examples.json\',examples);db.close()\n            result[\'audit_state_sha256\']=sha_file(work/\'audit_state.sqlite\')\n        if saved: saved.close()\n        if db is None and (work/\'audit_state.sqlite\').exists():\n            # A failed source/state check must never destroy the last good seal.\n            result[\'resume_state_untouched\'] = True\n            atomic_json(work/\'last_attempt.json\',result)\n        else:\n            atomic_json(work/\'summary.json\',result)\n    return result\n\n\ndef main():\n    p=argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--source\',type=Path,required=True);p.add_argument(\'--output\',type=Path,required=True)\n    a=p.parse_args();result=audit(a.source,a.output)\n    print(json.dumps(result,ensure_ascii=False,indent=2))\n    if result[\'status\']==\'PARENT_QC_AUDIT_FAILED\': raise SystemExit(1)\n\n\nif __name__==\'__main__\': main()\n'
EXPECTED_SCRIPT_SHA = 'd40f62527c1a7c556649d19c9e8968e7ba46194ded6cb84ea04f7752225bbc58'
assert hashlib.sha256(SCRIPT.encode()).hexdigest() == EXPECTED_SCRIPT_SHA
SCRIPT_PATH.write_bytes(SCRIPT.encode())
spec = importlib.util.spec_from_file_location('portoflow_parent_qc_audit', SCRIPT_PATH)
audit_tool = importlib.util.module_from_spec(spec)
spec.loader.exec_module(audit_tool)

if SOURCE is None:
    preferred = Path('/kaggle/input/notebooks/anhthu128/porto-flow/_output_.zip')
    if preferred.is_file():
        SOURCE = preferred
    else:
        candidates = list(Path('/kaggle/input').rglob('_output_.zip'))
        if len(candidates) != 1:
            raise ValueError('Attach ONLY the original matching Output, or set SOURCE to its _output_.zip. Candidates: '+str(candidates))
        SOURCE = candidates[0]
SOURCE = Path(SOURCE)

if RESUME is not None:
    resume = Path(RESUME).resolve()
    if WORK.exists() and any(WORK.iterdir()):
        raise ValueError('WORK already contains files; do not merge resume states')
    receipt = json.loads((resume/'summary.json').read_text())
    state = resume/'audit_state.sqlite'
    if receipt.get('source_zip_sha256') != audit_tool.SOURCE_SHA:
        raise ValueError('Resume belongs to another original Output')
    if state.stat().st_size > audit_tool.DEFAULT_LIMITS['output_bytes']:
        raise ValueError('Resume exceeds finite Output cap')
    if audit_tool.sha_file(state) != receipt.get('audit_state_sha256'):
        raise ValueError('Resume state checksum differs')
    WORK.mkdir(parents=True, exist_ok=True)
    required = state.stat().st_size+audit_tool.DEFAULT_LIMITS['reserve_bytes']
    if shutil.disk_usage(WORK).free < required:
        raise OSError('Insufficient free space for the small audit resume state')
    # Only this audit's bounded state/receipt are copied. No directory merge.
    shutil.copyfile(state,WORK/'audit_state.sqlite')
    shutil.copyfile(resume/'summary.json',WORK/'summary.json')

print('Original matching Input:', SOURCE, flush=True)
print('Finite budget:', audit_tool.DEFAULT_LIMITS, flush=True)


In [ ]:
result = audit_tool.audit(SOURCE, WORK)
print('STATUS:', result['status'], '| phase:', result['phase'], flush=True)
print('Batches:', result['committed_batches'], '/', result['available_batches'], flush=True)
print('Metrics:', json.dumps(result['metrics'], ensure_ascii=False, indent=2), flush=True)
if result.get('error'):
    print('STOP REASON:', result['error'], flush=True)
print('Coverage changed:', result['coverage_changed'], flush=True)

review = Path('/kaggle/working/portoflow_coverage_audit_review.zip')
with zipfile.ZipFile(review, 'w', zipfile.ZIP_DEFLATED) as archive:
    for name in ('summary.json','examples.json','budget.json','last_attempt.json'):
        path = WORK/name
        if path.is_file():
            if path.stat().st_size > 8*1024**2:
                raise ValueError('Small review member exceeds8MiB')
            archive.write(path,'portoflow_coverage_audit/'+name)
print('SEND FOR REVIEW:', review, '| bytes:', review.stat().st_size, flush=True)
print('RETAIN FULL Output:', WORK, '| contains bounded resumable audit_state.sqlite', flush=True)
if result['status'] == 'PARENT_QC_AUDIT_PARTIAL':
    print('PARTIAL: keep Output, attach it plus original matching Input, set RESUME to the small audit directory.', flush=True)
elif result['status'] == 'PARENT_QC_AUDIT_FAILED':
    raise RuntimeError('Audit failed validation; inspect STOP REASON and review ZIP. No matching or profile changes occurred.')
